# TP2 - Parte 1: dataset y preprocesamiento

Notebook de documentación y tratamiento del dataset **70 Dog Breeds Image Dataset**: análisis
exploratorio, control de calidad, conjunto independiente de evaluación, preprocesamiento,
data augmentation y splits (ver `ENUNCIADO.md`, sección 3).

Tiene que poder leerse como un informe: celdas Markdown que explican qué se hace y por qué,
código y salidas (gráficos y tablas) **ejecutadas y guardadas**. Una notebook sin salidas
visibles se considera no entregada.

Lo que produce esta notebook lo reutilizan `2-train.ipynb` y `train_classifier`: los splits
finales (CSV/JSON con rutas y etiquetas), el conjunto independiente y el preprocesamiento
encapsulado en un módulo de `src/lib/`.

**Notebook en Google Colab (si corresponde):** <reemplazar por el link, con permiso de lectura para los docentes>

Si entregan la notebook en Colab, esta copia del repositorio funciona como notebook de referencia:
el link de arriba es obligatorio y la notebook de Colab debe estar ejecutada, con sus salidas
(ver `ENUNCIADO.md`, sección 8.2).

## Equipo
- Alumno 1 :
- Alumno 2 :

In [4]:
import csv
import json
import time
import urllib.parse
import urllib.request
from pathlib import Path

# Carpeta donde guardaremos los resultados
CARPETA = Path("data/dataset/independent")
CARPETA.mkdir(parents=True, exist_ok=True)

# Nombre de la carpeta del dataset: término que buscaremos
razas = {
    "Afghan": "Afghan Hound",
    "Airedale": "Airedale Terrier",
    "Basset": "Basset Hound",
    "Beagle": "Beagle",
    "Border Collie": "Border Collie",
    "Boxer": "Boxer",
    "Bull Mastiff": "Bullmastiff",
    "Bulldog": "English Bulldog",
    "Chihuahua": "Chihuahua",
    "Dalmation": "Dalmatian",
    "Doberman": "Doberman",
    "French Bulldog": "French Bulldog",
    "German Sheperd": "German Shepherd",
    "Golden Retriever": "Golden Retriever",
    "Great Dane": "Great Dane",
    "Greyhound": "Greyhound",
    "Labrador": "Labrador Retriever",
    "Malinois": "Belgian Malinois",
    "Poodle": "Poodle",
    "Pug": "Pug"
}

URL_API = "https://api.openverse.org/v1/images/"

candidatas = []
ids_encontrados = set()

for carpeta, busqueda in razas.items():

    print(f"Buscando: {busqueda}...")

    parametros = {
        "q": busqueda,
        "page_size": 20,
        "license": "cc0,by,by-sa",
        "filter_dead": "true",
        "mature": "false"
    }

    url = URL_API + "?" + urllib.parse.urlencode(parametros)

    solicitud = urllib.request.Request(
        url,
        headers={"User-Agent": "TP-Computer-Vision/1.0"}
    )

    try:
        with urllib.request.urlopen(solicitud, timeout=30) as respuesta:
            datos = json.loads(respuesta.read().decode("utf-8"))

        resultados = datos.get("results", [])

        for imagen in resultados:

            identificador = imagen.get("id")

            # Evitar imágenes repetidas
            if identificador in ids_encontrados:
                continue

            ids_encontrados.add(identificador)

            candidatas.append({
                "seleccionada": "",
                "carpeta": carpeta,
                "busqueda": busqueda,
                "id": identificador,
                "titulo": imagen.get("title", ""),
                "autor": imagen.get("creator", ""),
                "autor_url": imagen.get("creator_url", ""),
                "licencia": imagen.get("license", ""),
                "version_licencia": imagen.get("license_version", ""),
                "licencia_url": imagen.get("license_url", ""),
                "fuente": imagen.get("source", ""),
                "pagina_original": imagen.get("foreign_landing_url", ""),
                "imagen_url": imagen.get("url", ""),
                "tipo_archivo": imagen.get("filetype", ""),
                "ancho": imagen.get("width", ""),
                "alto": imagen.get("height", ""),
                "atribucion": imagen.get("attribution", "")
            })

        print(f"  Resultados encontrados: {len(resultados)}")

    except Exception as error:
        print(f"  Error buscando {busqueda}: {error}")

    time.sleep(0.5)

# Exportar los resultados a CSV
archivo_csv = CARPETA / "candidatas.csv"

if candidatas:
    with open(archivo_csv, "w", newline="", encoding="utf-8-sig") as archivo:
        escritor = csv.DictWriter(
            archivo,
            fieldnames=candidatas[0].keys()
        )
        escritor.writeheader()
        escritor.writerows(candidatas)

print("\nBúsqueda terminada.")
print("Total de imágenes candidatas:", len(candidatas))
print("Archivo generado:", archivo_csv)

Buscando: Afghan Hound...
  Resultados encontrados: 20
Buscando: Airedale Terrier...
  Resultados encontrados: 20
Buscando: Basset Hound...
  Resultados encontrados: 20
Buscando: Beagle...
  Resultados encontrados: 20
Buscando: Border Collie...
  Resultados encontrados: 20
Buscando: Boxer...
  Resultados encontrados: 20
Buscando: Bullmastiff...
  Resultados encontrados: 20
Buscando: English Bulldog...
  Resultados encontrados: 20
Buscando: Chihuahua...
  Resultados encontrados: 20
Buscando: Dalmatian...
  Resultados encontrados: 20
Buscando: Doberman...
  Resultados encontrados: 20
Buscando: French Bulldog...
  Resultados encontrados: 20
Buscando: German Shepherd...
  Resultados encontrados: 20
Buscando: Golden Retriever...
  Resultados encontrados: 20
Buscando: Great Dane...
  Resultados encontrados: 20
Buscando: Greyhound...
  Resultados encontrados: 20
Buscando: Labrador Retriever...
  Resultados encontrados: 20
Buscando: Belgian Malinois...
  Resultados encontrados: 20
Buscando: Po

In [1]:

import pandas as pd
from IPython.display import display, HTML
from pathlib import Path
from html import escape

# 1. Cargar el archivo CSV
ruta_csv = Path("data/dataset/independent/candidatas.csv")
df = pd.read_csv(ruta_csv)

# 2. Comprobar que existan las columnas necesarias
columnas = [
    "carpeta", "titulo", "autor", "licencia",
    "imagen_url", "pagina_original"
]

faltantes = [c for c in columnas if c not in df.columns]

if faltantes:
    raise ValueError(f"Faltan columnas en el CSV: {faltantes}")

# 3. Construir la galería HTML
html = """
<style>
.galeria-raza {
    margin: 25px 0;
}
.galeria-grid {
    display: grid;
    grid-template-columns: repeat(auto-fill, minmax(190px, 1fr));
    gap: 14px;
}
.imagen-card {
    border: 1px solid #8885;
    border-radius: 9px;
    padding: 9px;
    overflow: hidden;
}
.imagen-card img {
    width: 100%;
    height: 145px;
    object-fit: contain;
    background: #eeeeee;
    border-radius: 5px;
}
.imagen-card p {
    font-size: 12px;
    overflow-wrap: anywhere;
}
.imagen-card a {
    font-size: 12px;
}
</style>
"""

# 4. Crear una sección por cada raza
for raza, grupo in df.groupby("carpeta", sort=True):

    html += f"""
    <section class="galeria-raza">
        <h2>{escape(str(raza))}</h2>
        <p>{len(grupo)} imágenes candidatas</p>
        <div class="galeria-grid">
    """

    for _, fila in grupo.iterrows():

        titulo = escape(str(fila["titulo"]))
        autor = escape(str(fila["autor"]))
        licencia = escape(str(fila["licencia"]))

        imagen = escape(str(fila["imagen_url"]), quote=True)
        original = escape(str(fila["pagina_original"]), quote=True)

        html += f"""
        <div class="imagen-card">
            <img
                src="{imagen}"
                loading="lazy"
                alt="{titulo}"
                onerror="this.style.display='none';
                this.nextElementSibling.style.display='block';"
            >
            <p style="display:none">No se pudo cargar la imagen</p>
            <p><b>{titulo}</b></p>
            <p>Autor: {autor}</p>
            <p>Licencia: {licencia}</p>
            <a href="{original}" target="_blank">
                Ver publicación original
            </a>
        </div>
        """

    html += """
        </div>
    </section>
    <hr>
    """

# 5. Mostrar la galería
display(HTML(html))


In [ ]:

import pandas as pd
import ipywidgets as widgets

from IPython.display import display, HTML
from pathlib import Path
from html import escape

# ==========================================
# 1. CONFIGURACIÓN
# ==========================================

ruta_csv = Path("data/dataset/independent/candidatas.csv")

df = pd.read_csv(ruta_csv)

MAX_POR_RAZA = {
    "Rottweiler": 6,
    "Siberian Husky": 6,
    "Shih-Tzu": 5
}

# Comprobar columnas necesarias
columnas = ["carpeta", "titulo", "autor", "licencia",
            "imagen_url", "pagina_original", "seleccionada"]

for columna in columnas:
    if columna not in df.columns:
        df[columna] = ""

# Normalizar selecciones anteriores
df["seleccionada"] = df["seleccionada"].fillna("").astype(str)

# Guardar automáticamente
def guardar_csv():
    df.to_csv(ruta_csv, index=False, encoding="utf-8-sig")

# Comprobar si una imagen ya está seleccionada
def esta_seleccionada(indice):
    return df.at[indice, "seleccionada"].strip().upper() in [
        "SI", "SÍ", "YES", "TRUE", "1"
    ]

# ==========================================
# 2. FUNCIONES DE SELECCIÓN
# ==========================================

contadores = {}
avisos = {}

def cantidad_seleccionada(raza):
    indices = df.index[df["carpeta"] == raza]
    return sum(esta_seleccionada(i) for i in indices)

def actualizar_estado(raza):
    cantidad = cantidad_seleccionada(raza)

    contadores[raza].value = (
        f"<b>{raza}:</b> {cantidad}/{MAX_POR_RAZA[raza]} imágenes seleccionadas"
    )

    # Al llegar a cinco, bloquear las restantes.
    for indice, checkbox in checkboxes[raza]:
        checkbox.disabled = (
            cantidad >= MAX_POR_RAZA[raza]
            and not esta_seleccionada(indice)
        )

def cambiar_seleccion(raza, indice, checkbox, cambio):

    if cambio["name"] != "value":
        return

    # Evitar superar el límite de cinco imágenes
    if cambio["new"] and cantidad_seleccionada(raza) >= MAX_POR_RAZA[raza]:
        checkbox.value = False
        avisos[raza].value = (
            "<span style='color:red'>"
            "Ya seleccionaste cinco imágenes para esta raza."
            "</span>"
        )
        return

    # Guardar selección
    df.at[indice, "seleccionada"] = (
        "SI" if cambio["new"] else ""
    )

    guardar_csv()

    avisos[raza].value = ""
    actualizar_estado(raza)

# ==========================================
# 3. CONSTRUIR GALERÍA
# ==========================================

checkboxes = {}

for raza, grupo in df.groupby("carpeta", sort=True):

    print(f"Preparando galería: {raza}")

    checkboxes[raza] = []

    titulo_raza = widgets.HTML(
        value=f"<h2>{escape(str(raza))}</h2>"
    )

    contador = widgets.HTML()
    contadores[raza] = contador

    aviso = widgets.HTML()
    avisos[raza] = aviso

    tarjetas = []

    for indice, fila in grupo.iterrows():

        titulo = escape(str(fila["titulo"]))
        autor = escape(str(fila["autor"]))
        licencia = escape(str(fila["licencia"]))

        imagen = escape(str(fila["imagen_url"]), quote=True)
        original = escape(str(fila["pagina_original"]), quote=True)

        contenido = widgets.HTML(
            value=f"""
            <div style="
                border:1px solid #8885;
                border-radius:8px;
                padding:8px;
                height:235px;
                overflow:hidden;
            ">
                <img
                    src="{imagen}"
                    loading="lazy"
                    style="
                        width:100%;
                        height:120px;
                        object-fit:contain;
                        background:#eeeeee;
                        border-radius:5px;
                    "
                    onerror="this.alt='Imagen no disponible';"
                >

                <p style="font-size:12px;">
                    <b>{titulo}</b><br>
                    Autor: {autor}<br>
                    Licencia: {licencia}<br>
                    <a href="{original}" target="_blank">
                        Ver fuente original
                    </a>
                </p>
            </div>
            """
        )

        checkbox = widgets.Checkbox(
            value=esta_seleccionada(indice),
            description="Seleccionar",
            indent=False,
            layout=widgets.Layout(width="100%")
        )

        checkbox.observe(
            lambda cambio, r=raza, i=indice, c=checkbox:
                cambiar_seleccion(r, i, c, cambio),
            names="value"
        )

        checkboxes[raza].append((indice, checkbox))

        tarjeta = widgets.VBox(
            [contenido, checkbox],
            layout=widgets.Layout(
                width="100%",
                padding="5px"
            )
        )

        tarjetas.append(tarjeta)

    # Organizar las imágenes en cuatro columnas
    galeria = widgets.GridBox(
        children=tarjetas,
        layout=widgets.Layout(
            grid_template_columns="repeat(4, minmax(0, 1fr))",
            grid_gap="12px",
            width="100%"
        )
    )

    actualizar_estado(raza)

    display(widgets.VBox([
        titulo_raza,
        contador,
        aviso,
        galeria
    ]))

    display(HTML("<hr>"))

# ==========================================
# 4. RESUMEN GENERAL
# ==========================================

total_seleccionadas = sum(
    esta_seleccionada(i) for i in df.index
)

print("Galería interactiva preparada.")
print("Imágenes seleccionadas:", total_seleccionadas)
print("Objetivo: 100 imágenes.")
print("Los cambios se guardan automáticamente.")


Preparando galería: Afghan


Preparando galería: Airedale


Preparando galería: Basset


Preparando galería: Beagle


Preparando galería: Border Collie


Preparando galería: Boxer


... (11 outputs left out)

Preparando galería: Malinois


Preparando galería: Poodle


Preparando galería: Pug


Galería interactiva preparada.
Imágenes seleccionadas: 0
Objetivo: 100 imágenes.
Los cambios se guardan automáticamente.


In [3]:
import pandas as pd

df = pd.read_csv(
    "data/dataset/independent/candidatas.csv"
)

seleccionadas = df[
    df["seleccionada"].astype(str).str.upper() == "SI"
]

print("Total seleccionadas:", len(seleccionadas))
print("\nImágenes por raza:")
print(seleccionadas["carpeta"].value_counts().sort_index())

Total seleccionadas: 83

Imágenes por raza:
carpeta
Afghan              5
Airedale            5
Basset              5
Border Collie       5
Bull Mastiff        5
Bulldog             5
Chihuahua           5
Dalmation           5
Doberman            5
French Bulldog      5
German Sheperd      5
Golden Retriever    5
Great Dane          5
Labrador            5
Malinois            5
Poodle              3
Pug                 5
Name: count, dtype: int64


In [4]:

import pandas as pd
import json
import urllib.parse
import urllib.request
import time

from pathlib import Path

ruta_csv = Path("data/dataset/independent/candidatas.csv")

# Leer el CSV existente y conservar las selecciones
df = pd.read_csv(ruta_csv)

nuevas_razas = {
    "Rottweiler": "Rottweiler",
    "Siberian Husky": "Siberian Husky",
    "Shih-Tzu": "Shih Tzu"
}

URL_API = "https://api.openverse.org/v1/images/"

ids_existentes = set(df["id"].dropna().astype(str))
nuevas_candidatas = []

for carpeta, busqueda in nuevas_razas.items():

    print(f"Buscando {busqueda}...")

    parametros = {
        "q": busqueda,
        "page_size": 20,
        "license": "cc0,by,by-sa",
        "filter_dead": "true",
        "mature": "false"
    }

    url = URL_API + "?" + urllib.parse.urlencode(parametros)

    solicitud = urllib.request.Request(
        url,
        headers={"User-Agent": "TP-Computer-Vision/1.0"}
    )

    try:
        with urllib.request.urlopen(solicitud, timeout=30) as respuesta:
            datos = json.loads(respuesta.read().decode("utf-8"))

        resultados = datos.get("results", [])

        agregadas = 0

        for imagen in resultados:

            identificador = str(imagen.get("id", ""))

            if not identificador or identificador in ids_existentes:
                continue

            ids_existentes.add(identificador)

            nuevas_candidatas.append({
                "seleccionada": "",
                "carpeta": carpeta,
                "busqueda": busqueda,
                "id": identificador,
                "titulo": imagen.get("title", ""),
                "autor": imagen.get("creator", ""),
                "autor_url": imagen.get("creator_url", ""),
                "licencia": imagen.get("license", ""),
                "version_licencia": imagen.get("license_version", ""),
                "licencia_url": imagen.get("license_url", ""),
                "fuente": imagen.get("source", ""),
                "pagina_original": imagen.get("foreign_landing_url", ""),
                "imagen_url": imagen.get("url", ""),
                "tipo_archivo": imagen.get("filetype", ""),
                "ancho": imagen.get("width", ""),
                "alto": imagen.get("height", ""),
                "atribucion": imagen.get("attribution", "")
            })

            agregadas += 1

        print(f"  Candidatas nuevas: {agregadas}")

    except Exception as error:
        print(f"  Error: {error}")

    time.sleep(0.5)

# Incorporar las nuevas filas sin modificar las anteriores
df_nuevas = pd.DataFrame(nuevas_candidatas)

df_final = pd.concat(
    [df, df_nuevas],
    ignore_index=True
)

df_final.to_csv(
    ruta_csv,
    index=False,
    encoding="utf-8-sig"
)

print("\nBúsqueda terminada.")
print("Nuevas candidatas:", len(df_nuevas))
print("Total de candidatas:", len(df_final))
print("Selecciones anteriores conservadas:",
      df_final["seleccionada"].fillna("").astype(str)
      .str.upper().eq("SI").sum())


Buscando Rottweiler...
  Candidatas nuevas: 20
Buscando Siberian Husky...
  Candidatas nuevas: 20
Buscando Shih Tzu...
  Candidatas nuevas: 20

Búsqueda terminada.
Nuevas candidatas: 60
Total de candidatas: 460
Selecciones anteriores conservadas: 83


## 1. Clonar el repositorio

Solo si trabajan en Colab. Si tu fork es privado, generá un token de acceso (GitHub → Settings →
Developer settings → Personal access tokens) y usalo en la URL, o subí un zip del proyecto a Colab/Drive.

In [ ]:
# TO-DO: completar con la URL de tu fork.
# Repo privado: https://<TOKEN>@github.com/<usuario>/tuia-dog-recognition-app.git
REPO_URL = "https://github.com/<usuario>/tuia-dog-recognition-app.git"

!git clone $REPO_URL proyecto
%cd proyecto

## 2. Instalar dependencias

Colab ya incluye torch, torchvision, opencv, numpy, scikit-learn y matplotlib;
solo se instala lo que falta.

In [ ]:
!pip install -q pydantic-settings python-dotenv kagglehub albumentations onnxruntime

## 3. Configuración del entorno

En Colab no hay PostgreSQL, por lo que se desactiva pgvector y se usa la base vectorial JSON.
La configuración se define por variables de entorno **antes** de importar `lib`.

In [ ]:
import os
import sys
from pathlib import Path

os.environ["USE_PGVECTOR"] = "false"
# Ajustá acá cualquier otra variable (IMAGE_SIZE, EMBEDDING_DIM, paths, etc.)

ROOT = Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

from lib.config import settings

import torch
print("dataset_path:", settings.dataset_path)
print("GPU disponible:", torch.cuda.is_available())

In [1]:
import sys
print(sys.executable)

import ipykernel
print(ipykernel.__version__)

c:\Users\U\Documents\python_proyectos\Python Proy\tuia-dog-recognition-app\.venv\Scripts\python.exe
7.4.0


In [3]:
import os

ruta = "data/dataset/train"

razas = sorted(os.listdir(ruta))

# Guardar todas las razas en un archivo de texto
with open("razas_dataset.txt", "w", encoding="utf-8") as archivo:
    archivo.write("\n".join(razas))

print("Total de razas:", len(razas))
print("\n".join(razas))


Total de razas: 70
Afghan
African Wild Dog
Airedale
American Hairless
American Spaniel
Basenji
Basset
Beagle
Bearded Collie
Bermaise
Bichon Frise
Blenheim
Bloodhound
Bluetick
Border Collie
Borzoi
Boston Terrier
Boxer
Bull Mastiff
Bull Terrier
Bulldog
Cairn
Chihuahua
Chinese Crested
Chow
Clumber
Cockapoo
Cocker
Collie
Corgi
Coyote
Dalmation
Dhole
Dingo
Doberman
Elk Hound
French Bulldog
German Sheperd
Golden Retriever
Great Dane
Great Perenees
Greyhound
Groenendael
Irish Spaniel
Irish Wolfhound
Japanese Spaniel
Komondor
Labradoodle
Labrador
Lhasa
Malinois
Maltese
Mex Hairless
Newfoundland
Pekinese
Pit Bull
Pomeranian
Poodle
Pug
Rhodesian
Rottweiler
Saint Bernard
Schnauzer
Scotch Terrier
Shar_Pei
Shiba Inu
Shih-Tzu
Siberian Husky
Vizsla
Yorkie


## 4. Descargar el dataset

In [ ]:
!python scripts/download_dataset.py
!ls data/dataset

## 5. Descripción del dataset

TO-DO:
- Cantidad de razas, imágenes por raza en cada split, formato, resolución y licencia de uso.
- Cómo están encuadradas las imágenes, comparado con los recortes de YOLO que va a recibir el
  clasificador en la Etapa 3.
- Limitaciones del dataset (razas parecidas, sesgos, etc.) y cómo pueden afectar los resultados.

In [ ]:
## TO-DO: descripción del dataset.

## 6. Análisis exploratorio (EDA)

TO-DO:
- Distribución de clases: imágenes por raza en cada split (gráfico y estadísticas). Desbalance.
- Resoluciones, relación de aspecto, canales, formatos y archivos corruptos.
- Grilla de ejemplos por raza, incluyendo pares de razas fáciles de confundir.
- Variaciones: iluminación, pose, fondo, oclusiones y cantidad de perros por imagen.

In [ ]:
## TO-DO: análisis exploratorio del dataset.

## 7. Control de calidad y filtrado

TO-DO: reglas explícitas y reproducibles (archivos corruptos, baja resolución, imágenes borrosas o
con exposición extrema, duplicados y casi-duplicados dentro y entre splits, imágenes mal etiquetadas).
Reportar cuántas imágenes descarta cada regla y mostrar ejemplos de descartes.

In [ ]:
## TO-DO: reglas de filtrado, conteo de descartes por regla y ejemplos.

## 8. Conjunto independiente de evaluación

TO-DO: imágenes que no pertenezcan al dataset (internet o fotos propias), con las mismas razas que
las carpetas del dataset. Incluir imágenes con un perro (Etapas 1 y 2), escenas con varios perros u
otros objetos e imágenes sin perros (Etapa 3). Documentar fuente y licencia; no subirlas al repositorio.

In [ ]:
## TO-DO: construcción y resumen del conjunto independiente (por ejemplo, en data/dataset/independent/<raza>/).

## 9. Preprocesamiento

TO-DO: definir y justificar el resize, la normalización y el orden de canales (la app entrega BGR).

⚠️ El mismo preprocesamiento debe usarse en las notebooks y en los servicios de la app:
encapsulalo en un módulo de `src/lib/` e importalo desde ambos lados.

In [ ]:
## TO-DO: transformaciones de preprocesamiento (importadas desde el módulo compartido).

## 10. Data augmentation

TO-DO: horizontal flip, rotación, blur, variaciones de brillo y contraste y ruido, con sus rangos
justificados. Mostrar una grilla de ejemplos aumentados de una misma imagen. Solo sobre `train`.

In [ ]:
## TO-DO: pipeline de data augmentation con ejemplos visuales.

## 11. Splits

TO-DO: definir train / validation / test (los del dataset o redefinidos, justificando), verificar
que no haya fuga de datos entre splits, fijar semillas y guardar los splits finales (CSV/JSON con
rutas y etiquetas) para `2-train.ipynb`.

In [ ]:
## TO-DO: splits finales, verificación de fuga de datos y exportación.

## 12. Resumen final

TO-DO: tabla con la cantidad de razas e imágenes por split, incluido el conjunto independiente.

In [ ]:
## TO-DO: tabla resumen.

## 13. Licencias y uso de imágenes

TO-DO: licencia del dataset, fuente y licencia de las imágenes del conjunto independiente y criterio
aplicado a imágenes con personas identificables.